In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram

In [2]:
def dj_oracle(case, n):
    oracle = QuantumCircuit(n + 1)
    if case == "balanced":
        b = np.random.randint(1, 2**n)            # non-zero b gives a balanced function
        b_str = format(b, '0' + str(n) + 'b')
        for q in range(n):
            if b_str[q] == '1': oracle.x(q)
        for q in range(n):
            oracle.cx(q, n)                       # CNOT from each input to ancilla
        for q in range(n):
            if b_str[q] == '1': oracle.x(q)
    elif case == "constant":
        if np.random.randint(2) == 1:
            oracle.x(n)                           # f = 1 everywhere
    return oracle.to_gate()

In [3]:
def dj_circuit(oracle_gate, n):
    qc = QuantumCircuit(n + 1, n)
    qc.x(n); qc.h(n)                              # ancilla |->
    for q in range(n):
        qc.h(q)
    qc.append(oracle_gate, range(n + 1))
    for q in range(n):
        qc.h(q)
    qc.measure(range(n), range(n))
    return qc

In [ ]:
backend = Aer.get_backend('qasm_simulator')
for case, n in [("constant", 8), ("balanced", 8), ("balanced", 10)]:
    qc = dj_circuit(dj_oracle(case, n), n)
    counts = backend.run(transpile(qc, backend), shots=1000).result().get_counts()
    print(case, n, "qubits:", counts)